In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast
from ast import literal_eval

In [2]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [3]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

/Users/elenanazarenko/opt/anaconda3/envs/py38/lib/python3.8/site-packages/spacy/util.py:877: UserWarning: [W095] Model 'de_core_news_sm' (3.0.0) was trained with spaCy v3.0 and may not be 100% compatible with the current version (3.5.0). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [4]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [5]:
def add_dots(row):
    list_out =[]
    if type(row)==str:
        row_list = row.split(" | ")
        for item in row_list:
            res = re.search(r"\...", item)
            if res is not None:
                list_out.append(item)
            else:
                list_out.append(item+"(...)")
        return " | ".join(list_out)

In [6]:
def clean_str(row):
    array_out =[]
    if len(row)==0:
        return list()
    else:
        array_out =[i.strip(' |') for i in row]
        
        return array_out

In [7]:
#function to split words and sentences into two different dataframes (currently doesn't used)
def split_words_sentences_short(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [8]:
#function to split words and sentences into two different dataframes for singular/plural dataframes
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []
    
    list_secondary_subcategory_word = []
    list_secondary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_secondary_subcategory_word.append(row["Secondary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_secondary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_secondary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [9]:
#function to separate one term binary pronouns excluding / (for example, exclude he/she)
def separate_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
    
    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)
        
        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word
        
    


In [10]:
#function to split words and sentences into two different dataframes for singular/plural dataframes. Used for gendered category to handle multiterms pronouns
def split_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []
    
    list_secondary_subcategory_word = []
    list_secondary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []
    
    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []
    
   
    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)
        
        
        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_secondary_subcategory_word.append(row["Secondary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
        
        elif type(span) == re.Match and len(word.rsplit("/"))==2:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_secondary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory","Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_secondary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [11]:
def replace_tolist(file_csv):

    # creating a variable and storing the search text
    search_text = "['']"
  
    # creating a variable and storing the replace text
    replace_text = "[]"
  
    # Opening our text file in read only
    # mode using the open() function
    with open(file_csv, 'r') as file:
        
        # Reading the content of the file
        # using the read() function and storing
        # them in a new variable
        data = file.read()
  
        # Searching and replacing the text
        # using the replace() function
        data = data.replace(search_text, replace_text)
    
    with open(file_csv, 'w') as file:
        # Writing the replaced data in our
        # text file
        file.write(data)
    return file_csv



In [12]:
# read csv file
df_rules = pd.read_csv("rules_01MAR_English_h.csv")

In [13]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2614 entries, 0 to 2613
Data columns (total 18 columns):
Lemma                     2613 non-null object
Word_Type                 2614 non-null object
Priority                  0 non-null float64
Category                  2614 non-null object
Primary_subcategory       2614 non-null object
Secondary_subcategory     721 non-null object
Alt_Field                 902 non-null object
Alt_Sg_Replacement        1481 non-null object
Identity_first            74 non-null object
Alt_Sg_people_first       243 non-null object
Alt_Sg_reframed           359 non-null object
Alt_Pl_collective_noun    66 non-null object
Alt_Pl                    432 non-null object
Identity_first_pl         29 non-null object
Alt_Pl_people_first       146 non-null object
Alt_Pl_reframed           81 non-null object
False_Positives           392 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(17)
memory usage: 367.7+ KB


In [14]:
df_rules[df_rules["Lemma"].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
243,NaN,acr,NaN,style,abbreviation,general,NaN,not applicable | not available,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(N/A),NaN


In [15]:
#convert NaN to string "N/A" in lemmas
df_rules.Lemma.fillna("N/A", inplace=True)

In [16]:
df_rules.iloc[[243]]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
243,N/A,acr,NaN,style,abbreviation,general,NaN,not applicable | not available,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,(N/A),NaN


### Drop duplicates

In [17]:
df_rules = df_rules.drop_duplicates()

In [18]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2612 entries, 0 to 2613
Data columns (total 18 columns):
Lemma                     2612 non-null object
Word_Type                 2612 non-null object
Priority                  0 non-null float64
Category                  2612 non-null object
Primary_subcategory       2612 non-null object
Secondary_subcategory     721 non-null object
Alt_Field                 900 non-null object
Alt_Sg_Replacement        1481 non-null object
Identity_first            74 non-null object
Alt_Sg_people_first       243 non-null object
Alt_Sg_reframed           359 non-null object
Alt_Pl_collective_noun    66 non-null object
Alt_Pl                    432 non-null object
Identity_first_pl         29 non-null object
Alt_Pl_people_first       146 non-null object
Alt_Pl_reframed           81 non-null object
False_Positives           392 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(17)
memory usage: 387.7+ KB


In [19]:
#check missing values in Primary subcategory
df_rules[df_rules['Word_Type'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives


In [20]:
#check missing values in Primary subcategory
df_rules[df_rules['Category'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives


In [21]:
# drop words with missing category
df_rules.dropna(subset=['Category'], inplace=True)

In [22]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2612 entries, 0 to 2613
Data columns (total 18 columns):
Lemma                     2612 non-null object
Word_Type                 2612 non-null object
Priority                  0 non-null float64
Category                  2612 non-null object
Primary_subcategory       2612 non-null object
Secondary_subcategory     721 non-null object
Alt_Field                 900 non-null object
Alt_Sg_Replacement        1481 non-null object
Identity_first            74 non-null object
Alt_Sg_people_first       243 non-null object
Alt_Sg_reframed           359 non-null object
Alt_Pl_collective_noun    66 non-null object
Alt_Pl                    432 non-null object
Identity_first_pl         29 non-null object
Alt_Pl_people_first       146 non-null object
Alt_Pl_reframed           81 non-null object
False_Positives           392 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(17)
memory usage: 387.7+ KB


### Check missing values for word type

In [23]:
#check missing values in Primary subcategory
df_rules[df_rules['Word_Type'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives


### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing

In [24]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives


In [25]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [26]:
df_rules[df_rules["Primary_subcategory"]=="Nazi_language"]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives


In [27]:
df_rules["Primary_subcategory"].unique()

array(['abbreviation', 'behavior', 'medical_state', 'homophobia',
       'vision', 'ability', 'racism', 'antisemitism', 'belief',
       'antimuslim', 'sports_terms', 'gender_identity', 'titles',
       'function', 'racist_source', 'xenophobia', 'hollow', 'filler',
       'cognitive_perception', 'female_stereotype', 'formality', 'sexism',
       'offensive_language', 'sexual_orientation', 'agentic', 'classism',
       'learning', 'exaggerating', 'emotional_security', 'd_and_i',
       'communal', 'age_old', 'male_stereotype', 'military_source',
       'hearing', 'cognitive_ability', 'leadership', 'mental_wellbeing',
       'color', 'hidden_image', 'mobility', 'transphobia', 'migration',
       'age_young', 'culture', 'speech', 'ableism', 'binary_pronouns'],
      dtype=object)

In [28]:
df_rules = df_rules[df_rules["Primary_subcategory"]!="Nazi_language"]

In [29]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2612 entries, 0 to 2613
Data columns (total 18 columns):
Lemma                     2612 non-null object
Word_Type                 2612 non-null object
Priority                  0 non-null float64
Category                  2612 non-null object
Primary_subcategory       2612 non-null object
Secondary_subcategory     721 non-null object
Alt_Field                 900 non-null object
Alt_Sg_Replacement        1481 non-null object
Identity_first            74 non-null object
Alt_Sg_people_first       243 non-null object
Alt_Sg_reframed           359 non-null object
Alt_Pl_collective_noun    66 non-null object
Alt_Pl                    432 non-null object
Identity_first_pl         29 non-null object
Alt_Pl_people_first       146 non-null object
Alt_Pl_reframed           81 non-null object
False_Positives           392 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(17)
memory usage: 387.7+ KB


In [30]:
# Check Identity_first alternatives
df_rules[df_rules['Identity_first'].notna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
3,ADD,-abbr,NaN,unconscious_bias,behavior,abbreviation,NaN,attention deficit hyperactivity disability (AD...,neurodivergent | ADHDer,person with ADHD | someone who has ADHD,easily bored | easily distracted | having a po...,NaN,NaN,NaN,NaN,NaN,(ADHD) | ADHDer,NaN
23,Asperger,s,NaN,unconscious_bias,ability,NaN,"--- only if the person prefers the term, or wh...",on the autism spectrum,Autistic | neurodivergent,person who has autism | someone with autism,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24,Asperger’s Syndrome,s,NaN,unconscious_bias,ability,NaN,"--- only if the person prefers the term, or wh...",being on the autism spectrum,Autistic | neurodivergent,person who has autism | someone with autism,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,PDD NOS,acr,NaN,unconscious_bias,behavior,NaN,NaN,pervasive developmental disorder not otherwise...,Autistic | neurodivergent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
272,PDD-NOS,acr,NaN,unconscious_bias,behavior,NaN,NaN,pervasive developmental disorder not otherwise...,Autistic | neurodivergent,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2533,visually-challenged,=a,NaN,unconscious_bias,vision,NaN,NaN,blind | visually impaired | vision-impaired,Blind,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2534,visuallychallenged,=a,NaN,unconscious_bias,vision,NaN,NaN,blind | visually impaired | vision-impaired,Blind,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2582,with blindness,-i,NaN,unconscious_bias,vision,NaN,NaN,who is blind visually impaired | who is vision...,Blind | who is Blind,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2583,with deafness,i,NaN,unconscious_bias,hearing,NaN,NaN,who is hard-of-hearing,Deaf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
# Check Identity_first alternatives
df_rules[df_rules['Identity_first_pl'].notna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives
597,birth defect,s,NaN,unconscious_bias,ability,NaN,---name congenital health condition or disability,has a disability | congenital disability | has...,is disabled,NaN,NaN,NaN,having disabilities | congenital disabilities ...,are disabled,NaN,NaN,NaN,NaN
598,birth-defect,s,NaN,unconscious_bias,ability,NaN,---name congenital health condition or disability,has a disability | congenital disability | has...,is disabled,NaN,NaN,NaN,having disabilities | congenital disabilities ...,are disabled,NaN,NaN,NaN,NaN
599,birthdefect,s,NaN,unconscious_bias,ability,NaN,---name congenital health condition or disability,has a disability | congenital disability | has...,is disabled,NaN,NaN,NaN,having disabilities | congenital disabilities ...,are disabled,NaN,NaN,NaN,NaN
781,coexisting difficulty,i,NaN,unconscious_bias,ability,NaN,NaN,co-occurring disability | concurrent disabilit...,disabled,NaN,NaN,NaN,co-occurring disabilities | concurrent disabil...,disabled,NaN,NaN,NaN,NaN
782,cognitive challenge,i,NaN,unconscious_bias,cognitive_perception,NaN,NaN,cognitive disability | cognition | cognitive s...,cognitively disabled | intellectually disabled,NaN,NaN,NaN,cognitive disability | cognition | cognitive s...,cognitively disabled | intellectually disabled,NaN,NaN,NaN,NaN
1514,intellectual disorder,i,NaN,unconscious_bias,cognitive_perception,NaN,--- name the specific disability,intellectual disability,intellectually disabled,NaN,NaN,NaN,intellectual disabilities,intellectually disabled,NaN,NaN,NaN,NaN
1622,language based learning difficulty,s,NaN,unconscious_bias,learning,NaN,--- name the specific disability or condition,language-based learning disability,learning disabled,NaN,NaN,NaN,language-based learning disabilities,learning disabled,NaN,NaN,NaN,NaN
1625,language-based learning difficulty,s,NaN,unconscious_bias,learning,NaN,--- name the specific disability or condition,language-based learning disability,learning disabled,NaN,NaN,NaN,language-based learning disabilities,learning disabled,NaN,NaN,NaN,NaN
1626,languagebased learning difficulty,s,NaN,unconscious_bias,learning,NaN,--- name the specific disability or condition,language-based learning disability,learning disabled,NaN,NaN,NaN,language-based learning disabilities,learning disabled,NaN,NaN,NaN,NaN
1639,learning difference,=s,NaN,unconscious_bias,learning,NaN,--- name the specific disability or condition,learning disability,learning disabled,NaN,NaN,NaN,learning disabilities,learning disabled,NaN,NaN,NaN,NaN


### add (...) to 
Refraimed columns:
Alt_Pl_people_first
Alt_Pl_reframed 

Alt_Sg_people_first
Alt_Sg_reframed

### functions tests

In [32]:
test = "in einem Fach bewanderte Person | jemand vom Fach | fachlich versierte Person | Person, die sich (auf ...) spezialisiert hat |"

In [33]:
test_1 = ["in einem Fach bewanderte Person |", " jemand vom Fach |", " fachlich versierte Person", "| Person, die sich (auf ...) spezialisiert hat |"]

In [34]:
test_2 = add_dots(test)

In [35]:
test_4 = []

In [36]:
# add (...) for inspiration alternatives
df_rules["Alt_Pl_reframed"]=df_rules["Alt_Pl_reframed"].map(add_dots)

In [37]:
# add (...) for inspiration alternatives
df_rules["Alt_Pl_people_first"]=df_rules["Alt_Pl_people_first"].map(add_dots)

In [38]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_people_first"]=df_rules["Alt_Sg_people_first"].map(add_dots)

In [39]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_reframed"]=df_rules["Alt_Sg_reframed"].map(add_dots)

In [40]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2612 entries, 0 to 2613
Data columns (total 18 columns):
Lemma                     2612 non-null object
Word_Type                 2612 non-null object
Priority                  0 non-null float64
Category                  2612 non-null object
Primary_subcategory       2612 non-null object
Secondary_subcategory     721 non-null object
Alt_Field                 900 non-null object
Alt_Sg_Replacement        1481 non-null object
Identity_first            74 non-null object
Alt_Sg_people_first       243 non-null object
Alt_Sg_reframed           359 non-null object
Alt_Pl_collective_noun    66 non-null object
Alt_Pl                    432 non-null object
Identity_first_pl         29 non-null object
Alt_Pl_people_first       146 non-null object
Alt_Pl_reframed           81 non-null object
False_Positives           392 non-null object
3rd_party_alternatives    2 non-null object
dtypes: float64(1), object(17)
memory usage: 387.7+ KB


In [41]:
test2= df_rules.query("Lemma=='endurance'")

In [42]:
test2["Alt_Sg_Replacement"]

1048    resilience | bearing | tolerance | forbearance...
Name: Alt_Sg_Replacement, dtype: object

In [43]:
pd.set_option('display.max_rows', None)

In [44]:
test2["Alt_Sg_Replacement"]

1048    resilience | bearing | tolerance | forbearance...
Name: Alt_Sg_Replacement, dtype: object

In [45]:
test2["Alt_Sg_reframed"]

1048    ability to carry on(...) | capacity to keep at...
Name: Alt_Sg_reframed, dtype: object

## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [46]:
#order for single form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Identity_first", "Alt_Sg_Replacement", "Alt_Field", "Alt_Pl_collective_noun", "Alt_Pl", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [47]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

In [48]:
df_rules.query("Lemma=='endurance'")

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,Alt_Sg_reframed,Alt_Pl_collective_noun,Alt_Pl,Identity_first_pl,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives,Sg_all,Sg_all_split
1048,endurance,s,NaN,unconscious_bias,agentic,competitive,NaN,resilience | bearing | tolerance | forbearance...,NaN,None,ability to carry on(...) | capacity to keep at...,NaN,NaN,NaN,None,None,NaN,NaN,resilience | bearing | tolerance | forbearance...,"[resilience, bearing, tolerance, forbearance, ..."


## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [49]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Identity_first_pl", "Alt_Pl", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first", "Alt_Pl_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [50]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

### Merge alternatives into one column

In [51]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [52]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives']= df_rules.iloc[:,source_col_loc:source_col_loc+8].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [53]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [54]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Identity_first,Alt_Sg_people_first,...,Alt_Pl_people_first,Alt_Pl_reframed,False_Positives,3rd_party_alternatives,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,(x)aaS,acr,NaN,style,abbreviation,marketing,NaN,something as a service | something as a servic...,NaN,None,...,None,None,((x)aaS),NaN,something as a service | something as a servic...,"[something as a service, something as a servic...",,[],something as a service | something as a servic...,"[something as a service, something as a servic..."
1,A.M.,abbr,NaN,style,abbreviation,job_department_titles,NaN,Master of Arts | Master of Arts (A.M.),NaN,None,...,None,None,(A.M.),NaN,Master of Arts | Master of Arts (A.M.),"[Master of Arts, Master of Arts (A.M.)]",,[],Master of Arts | Master of Arts (A.M.),"[Master of Arts, Master of Arts (A.M.)]"
2,ACCT,abbr,NaN,style,abbreviation,finance,NaN,account,NaN,None,...,None,None,(ACCT),NaN,account,[account],,[],account,[account]
3,ADD,-abbr,NaN,unconscious_bias,behavior,abbreviation,NaN,attention deficit hyperactivity disability (AD...,neurodivergent | ADHDer,person with ADHD(...) | someone who has ADHD(...),...,None,None,(ADHD) | ADHDer,NaN,neurodivergent | ADHDer | attention deficit hy...,"[neurodivergent, ADHDer, attention deficit hyp...",,[],attention deficit hyperactivity disability (AD...,[attention deficit hyperactivity disability (A...
4,ADHD,abbr,NaN,style,abbreviation,medical,NaN,attention deficit hyperactivity disorder | att...,NaN,person with attention deficit hyperactivity di...,...,people with attention deficit hyperactivity di...,fidgety(...) | restless(...) | easily bored(...),(ADHD),NaN,attention deficit hyperactivity disorder | att...,"[attention deficit hyperactivity disorder, att...",people with attention deficit hyperactivity di...,[people with attention deficit hyperactivity d...,attention deficit hyperactivity disorder | att...,"[attention deficit hyperactivity disorder, att..."


### Cleaning of alternatives

In [55]:
# strip alternatives 
df_rules["Alt_split"]=df_rules["Alt_split"].map(clean_str)

In [56]:
# strip alternatives 
df_rules["Sg_all_split"]=df_rules["Sg_all_split"].map(clean_str)

In [57]:
# strip alternatives 
df_rules["Pl_all_split"]=df_rules["Pl_all_split"].map(clean_str)

### Create subsets of dataframes

In [58]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"]]

In [59]:
df_rules[df_rules["Word_Type"]=="=v"]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
612,blacklisting,=v,unconscious_bias,color,cultural,"[blocklisting, denylisting, excluding, stoplis...",,[],"[blocklisting, denylisting, excluding, stoplis..."


In [60]:
df_rules[df_rules["Word_Type"]=="-v"]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [61]:
df_verbs = df_rules[(df_rules["Word_Type"]=="v") | (df_rules["Word_Type"]=="=v")]

In [62]:
df_verbs.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 98 entries, 415 to 2606
Data columns (total 9 columns):
Lemma                    98 non-null object
Word_Type                98 non-null object
Category                 98 non-null object
Primary_subcategory      98 non-null object
Secondary_subcategory    64 non-null object
Sg_all_split             98 non-null object
Pl_all                   98 non-null object
Pl_all_split             98 non-null object
Alt_split                98 non-null object
dtypes: object(9)
memory usage: 7.7+ KB


In [63]:
df_verbs["Category"].unique()

array(['unconscious_bias', 'inclusive', 'style', 'openly_discriminating',
       'gendered'], dtype=object)

In [64]:
df_verbs[df_verbs["Category"]=='style']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
574,believe me,v,style,filler,NaN,[-],-,[-],[-]
1653,leverage,v,style,hollow,NaN,"[use, capitalize on]",,[],"[use, capitalize on]"
2208,seem,v,style,filler,NaN,[-],-,[-],[-]
2382,surpass,v,style,exaggerating,NaN,"[refine, improve upon, transcend, do better th...",,[],"[refine, improve upon, transcend, do better th..."
2395,synergize,v,style,hollow,NaN,[work together],,[],[work together]
2521,utilize,v,style,hollow,NaN,[use],,[],[use]


### Check duplicates&homonyms

In [65]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
735,challenge,v,unconscious_bias,agentic,competitive,"[question, cross-examine, re-evaluate, object ...",,[],"[question, cross-examine, re-evaluate, object ..."
736,challenge,s,unconscious_bias,agentic,competitive,"[test, trial, burning question, complex situat...",complex situations | difficult tasks | burning...,"[complex situations, difficult tasks, burning ...","[test, trial, burning question, complex situat..."
944,direct,v,unconscious_bias,agentic,rough,"[guide, show, point, organize, encourage, assi...",,[],"[guide, show, point, organize, encourage, assi..."
945,direct,a,unconscious_bias,agentic,rough,"[straightforward, open, candid, --- don't use ...",--- don't use for people,[--- don't use for people],"[--- don't use for people, straightforward, op..."
1006,drive,s,unconscious_bias,agentic,NaN,"[spirit, energy, motivation, appetite, growth ...",,[],"[spirit, energy, motivation, appetite, growth ..."
1007,drive,v,unconscious_bias,agentic,competitive,"[invest in, support, work toward, take respons...",,[],"[invest in, support, work toward, take respons..."
1016,dummy,s,unconscious_bias,ability,NaN,"[placeholder, anthropomorphic test device, reh...",,[],"[placeholder, anthropomorphic test device, reh..."
1017,dummy,a,unconscious_bias,ability,NaN,"[unused, non-functioning, base, fake, mock, stub]",,[],"[unused, non-functioning, base, fake, mock, stub]"
1022,dwarf,s,unconscious_bias,medical_state,NaN,"[person with restricted growth(...), person of...",persons with restricted growth(...) | persons ...,"[persons with restricted growth(...), persons ...","[person with restricted growth(...), person of..."
1023,dwarf,v,unconscious_bias,medical_state,NaN,"[tower over, loom over, do better than, stunt,...",,[],"[tower over, loom over, do better than, stunt,..."


### Homonyms

In [66]:
# check how many homonyms
df_rules["Lemma"].duplicated(keep = False).sum()

58

In [67]:
df_homonyms = df_rules[df_rules["Lemma"].duplicated(keep = False)]

In [68]:
#check first 5 rows
df_homonyms

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
735,challenge,v,unconscious_bias,agentic,competitive,"[question, cross-examine, re-evaluate, object ...",,[],"[question, cross-examine, re-evaluate, object ..."
736,challenge,s,unconscious_bias,agentic,competitive,"[test, trial, burning question, complex situat...",complex situations | difficult tasks | burning...,"[complex situations, difficult tasks, burning ...","[test, trial, burning question, complex situat..."
944,direct,v,unconscious_bias,agentic,rough,"[guide, show, point, organize, encourage, assi...",,[],"[guide, show, point, organize, encourage, assi..."
945,direct,a,unconscious_bias,agentic,rough,"[straightforward, open, candid, --- don't use ...",--- don't use for people,[--- don't use for people],"[--- don't use for people, straightforward, op..."
1006,drive,s,unconscious_bias,agentic,NaN,"[spirit, energy, motivation, appetite, growth ...",,[],"[spirit, energy, motivation, appetite, growth ..."
1007,drive,v,unconscious_bias,agentic,competitive,"[invest in, support, work toward, take respons...",,[],"[invest in, support, work toward, take respons..."
1016,dummy,s,unconscious_bias,ability,NaN,"[placeholder, anthropomorphic test device, reh...",,[],"[placeholder, anthropomorphic test device, reh..."
1017,dummy,a,unconscious_bias,ability,NaN,"[unused, non-functioning, base, fake, mock, stub]",,[],"[unused, non-functioning, base, fake, mock, stub]"
1022,dwarf,s,unconscious_bias,medical_state,NaN,"[person with restricted growth(...), person of...",persons with restricted growth(...) | persons ...,"[persons with restricted growth(...), persons ...","[person with restricted growth(...), person of..."
1023,dwarf,v,unconscious_bias,medical_state,NaN,"[tower over, loom over, do better than, stunt,...",,[],"[tower over, loom over, do better than, stunt,..."


In [69]:
#create subset of dataframe to drop unnecessary columns
df_homonyms = df_homonyms[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [70]:
df_homonyms.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 58 entries, 735 to 2571
Data columns (total 5 columns):
Lemma                  58 non-null object
Word_Type              58 non-null object
Category               58 non-null object
Primary_subcategory    58 non-null object
Alt_split              58 non-null object
dtypes: object(5)
memory usage: 2.7+ KB


In [71]:
#save homonyms to csv file after preprocessing
df_homonyms.to_csv("homonyms_words.csv", index=False)

In [72]:
replace_tolist(r"homonyms_words.csv")

'homonyms_words.csv'

### Remove homonyms from dataframe for the next steps pre-processing

In [73]:
homonyms_list = list(df_rules[df_rules["Lemma"].duplicated()]["Lemma"])

In [74]:
homonyms_list

['challenge',
 'direct',
 'drive',
 'dummy',
 'dwarf',
 'elderly',
 'female',
 'fight',
 'force',
 'handicap',
 'hysteric',
 'impact',
 'lead',
 'male',
 'man',
 'master',
 'must-have',
 'musthave',
 'nag',
 'ninja',
 'objective',
 'pioneer',
 'psycho',
 'queer',
 'sass',
 'schizo',
 'stupid',
 'wacko',
 'whitewash']

In [75]:
df_rules = df_rules.query("Lemma != @homonyms_list")

### Binary_pronouns

In [76]:
df_binary_pronouns = df_rules[df_rules["Primary_subcategory"]=="binary_pronouns"]

In [77]:
df_binary_pronouns.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 27 entries, 1362 to 2252
Data columns (total 9 columns):
Lemma                    27 non-null object
Word_Type                27 non-null object
Category                 27 non-null object
Primary_subcategory      27 non-null object
Secondary_subcategory    0 non-null object
Sg_all_split             27 non-null object
Pl_all                   27 non-null object
Pl_all_split             27 non-null object
Alt_split                27 non-null object
dtypes: object(9)
memory usage: 2.1+ KB


In [78]:
df_binary_pronouns.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
1362,he,s,gendered,binary_pronouns,NaN,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."
1363,he or she,s,gendered,binary_pronouns,NaN,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."
1364,he/she,s,gendered,binary_pronouns,NaN,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."
1375,her,s,gendered,binary_pronouns,NaN,"[them, their, ((given name)), ((given name fam...",,[],"[them, their, ((given name)), ((given name fam..."
1376,her or him,s,gendered,binary_pronouns,NaN,"[them, ((given name)), ((given name family nam...",,[],"[them, ((given name)), ((given name family nam..."


In [79]:
# separate one term binary pronouns 
df_binary_pronouns_word =separate_binary_pronouns(df_binary_pronouns)

In [80]:
#check the results
df_binary_pronouns_word

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,he,s,gendered,binary_pronouns,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."
1,her,s,gendered,binary_pronouns,"[them, their, ((given name)), ((given name fam...",,[],"[them, their, ((given name)), ((given name fam..."
2,hers,s,gendered,binary_pronouns,"[theirs, ((given name's))]",,[],"[theirs, ((given name's))]"
3,herself,s,gendered,binary_pronouns,"[themself, theirself, self]",,[],"[themself, theirself, self]"
4,him,s,gendered,binary_pronouns,"[them, ((given name)), ((given name family nam...",,[],"[them, ((given name)), ((given name family nam..."
5,himself,s,gendered,binary_pronouns,"[themself, theirself, self]",,[],"[themself, theirself, self]"
6,his,s,gendered,binary_pronouns,"[their, ((given name's)), theirs]",,[],"[their, ((given name's)), theirs]"
7,she,s,gendered,binary_pronouns,"[they, ((given name)), ((given name family nam...",,[],"[they, ((given name)), ((given name family nam..."


In [81]:
#create subset of dataframe to drop unnecessary columns
df_binary_pronouns_word = df_binary_pronouns_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all_split", "Alt_split"]]

In [82]:
#save one term binary_pronouns to csv file after preprocessing
df_binary_pronouns_word.to_csv("ub_singular_they.csv", index=False)

In [83]:
df_binary_pronouns_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
Lemma                  8 non-null object
Word_Type              8 non-null object
Category               8 non-null object
Primary_subcategory    8 non-null object
Sg_all_split           8 non-null object
Pl_all_split           8 non-null object
Alt_split              8 non-null object
dtypes: object(7)
memory usage: 576.0+ bytes


In [84]:
replace_tolist(r"ub_singular_they.csv")

'ub_singular_they.csv'

In [85]:
#test read of dataframe
df_test = pd.read_csv("ub_singular_they.csv")

In [86]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
Lemma                  8 non-null object
Word_Type              8 non-null object
Category               8 non-null object
Primary_subcategory    8 non-null object
Sg_all_split           8 non-null object
Pl_all_split           8 non-null object
Alt_split              8 non-null object
dtypes: object(7)
memory usage: 576.0+ bytes


In [87]:
df_test

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split
0,he,s,gendered,binary_pronouns,"['they', '((given name))', '((given name famil...",[],"['they', '((given name))', '((given name famil..."
1,her,s,gendered,binary_pronouns,"['them', 'their', '((given name))', '((given n...",[],"['them', 'their', '((given name))', '((given n..."
2,hers,s,gendered,binary_pronouns,"['theirs', ""((given name's))""]",[],"['theirs', ""((given name's))""]"
3,herself,s,gendered,binary_pronouns,"['themself', 'theirself', 'self']",[],"['themself', 'theirself', 'self']"
4,him,s,gendered,binary_pronouns,"['them', '((given name))', '((given name famil...",[],"['them', '((given name))', '((given name famil..."
5,himself,s,gendered,binary_pronouns,"['themself', 'theirself', 'self']",[],"['themself', 'theirself', 'self']"
6,his,s,gendered,binary_pronouns,"['their', ""((given name's))"", 'theirs']",[],"['their', ""((given name's))"", 'theirs']"
7,she,s,gendered,binary_pronouns,"['they', '((given name))', '((given name famil...",[],"['they', '((given name))', '((given name famil..."


### Remove one term binary_pronouns from dataframe for the next steps pre-processing

In [88]:
#remove binary pronouns
binary_list = list(df_binary_pronouns_word["Lemma"])

In [89]:
len(binary_list)

8

In [90]:
df_rules = df_rules.query("Lemma != @binary_list")

In [91]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2546 entries, 0 to 2613
Data columns (total 9 columns):
Lemma                    2546 non-null object
Word_Type                2546 non-null object
Category                 2546 non-null object
Primary_subcategory      2546 non-null object
Secondary_subcategory    705 non-null object
Sg_all_split             2546 non-null object
Pl_all                   2546 non-null object
Pl_all_split             2546 non-null object
Alt_split                2546 non-null object
dtypes: object(9)
memory usage: 198.9+ KB


### Acronyms (abbreviations)

In [92]:
#check word_type
df_rules["Word_Type"].unique()

array(['acr', 'abbr', '-abbr', 's', 'i', 'a', '=s', '-acr', 'adv', '=a',
       'v', 'conj', '=v', '-a', '-i'], dtype=object)

In [93]:
df_abbr = df_rules[(df_rules["Word_Type"]=='abbr') | (df_rules["Word_Type"]=='acr') | (df_rules["Word_Type"]=='-acr') | (df_rules["Word_Type"]=='-abbr')]

In [94]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 318 entries, 0 to 1998
Data columns (total 9 columns):
Lemma                    318 non-null object
Word_Type                318 non-null object
Category                 318 non-null object
Primary_subcategory      318 non-null object
Secondary_subcategory    302 non-null object
Sg_all_split             318 non-null object
Pl_all                   318 non-null object
Pl_all_split             318 non-null object
Alt_split                318 non-null object
dtypes: object(9)
memory usage: 24.8+ KB


In [95]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [96]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations.csv", index=False)

In [97]:
replace_tolist(r"abbreviations.csv")

'abbreviations.csv'

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [98]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [99]:
df_rules = df_rules.query("Lemma != @abbr_list")

In [100]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2228 entries, 6 to 2613
Data columns (total 9 columns):
Lemma                    2228 non-null object
Word_Type                2228 non-null object
Category                 2228 non-null object
Primary_subcategory      2228 non-null object
Secondary_subcategory    403 non-null object
Sg_all_split             2228 non-null object
Pl_all                   2228 non-null object
Pl_all_split             2228 non-null object
Alt_split                2228 non-null object
dtypes: object(9)
memory usage: 174.1+ KB


## Openly_discriminating

In [101]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [102]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 568 entries, 21 to 2605
Data columns (total 9 columns):
Lemma                    568 non-null object
Word_Type                568 non-null object
Category                 568 non-null object
Primary_subcategory      568 non-null object
Secondary_subcategory    0 non-null object
Sg_all_split             568 non-null object
Pl_all                   568 non-null object
Pl_all_split             568 non-null object
Alt_split                568 non-null object
dtypes: object(9)
memory usage: 44.4+ KB


In [103]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
21,African queen,s,openly_discriminating,homophobia,NaN,[-],-,[-],[-]
25,Aunt Jemima,s,openly_discriminating,racism,NaN,[-],-,[-],[-]
38,Barbie,s,openly_discriminating,racism,NaN,[-],-,[-],[-]
39,Beastie Boy,s,openly_discriminating,antisemitism,NaN,[-],-,[-],[-]
40,Beulah,s,openly_discriminating,racism,NaN,[-],-,[-],[-]


In [104]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [105]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Barbie,s,openly_discriminating,racism,NaN,[-],-,[-],[-]
1,Beulah,s,openly_discriminating,racism,NaN,[-],-,[-],[-]
2,Bin-Laden,s,openly_discriminating,antimuslim,NaN,[-],-,[-],[-]
3,Binladen,s,openly_discriminating,antimuslim,NaN,[-],-,[-],[-]
4,Brownie,s,openly_discriminating,homophobia,NaN,[-],-,[-],[-]


In [106]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [107]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words.csv", index=False)

In [108]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences.csv", index=False)

In [109]:
replace_tolist(r"open_dis_words.csv")

'open_dis_words.csv'

In [110]:
replace_tolist(r"open_dis_sentences.csv")

'open_dis_sentences.csv'

## Style

In [111]:
df_style = df_rules[df_rules["Category"]=="style"]

In [112]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 232 entries, 140 to 2609
Data columns (total 9 columns):
Lemma                    232 non-null object
Word_Type                232 non-null object
Category                 232 non-null object
Primary_subcategory      232 non-null object
Secondary_subcategory    8 non-null object
Sg_all_split             232 non-null object
Pl_all                   232 non-null object
Pl_all_split             232 non-null object
Alt_split                232 non-null object
dtypes: object(9)
memory usage: 18.1+ KB


In [113]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [114]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [115]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words.csv", index=False)

In [116]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences.csv", index=False)

In [117]:
replace_tolist(r"style_words.csv")

'style_words.csv'

In [118]:
replace_tolist(r"style_sentences.csv")

'style_sentences.csv'

## Inclusive

In [119]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [120]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 269 entries, 419 to 2606
Data columns (total 9 columns):
Lemma                    269 non-null object
Word_Type                269 non-null object
Category                 269 non-null object
Primary_subcategory      269 non-null object
Secondary_subcategory    125 non-null object
Sg_all_split             269 non-null object
Pl_all                   269 non-null object
Pl_all_split             269 non-null object
Alt_split                269 non-null object
dtypes: object(9)
memory usage: 21.0+ KB


In [121]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
419,absorbed,=a,inclusive,emotional_security,NaN,[],,[],[]
423,accessibility,s,inclusive,d_and_i,NaN,[],,[],[]
433,affectionate,a,inclusive,communal,caring,[],,[],[]
434,affinity group,s,inclusive,d_and_i,NaN,[],,[],[]
440,ages,s,inclusive,d_and_i,NaN,[],,[],[]


In [122]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [123]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [124]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 4 columns):
Lemma                  26 non-null object
Word_Type              26 non-null object
Category               26 non-null object
Primary_subcategory    26 non-null object
dtypes: object(4)
memory usage: 960.0+ bytes


In [125]:
#save to csv file after preprocessing
df_inclusive_word.to_csv("inclusive_words.csv", index=False)

In [126]:
#save to csv file after preprocessing
df_inclusive_sentence.to_csv("inclusive_sentences.csv", index=False)

In [127]:
replace_tolist(r"inclusive_words.csv")

'inclusive_words.csv'

In [128]:
replace_tolist(r"inclusive_sentences.csv")

'inclusive_sentences.csv'

## Gendered

In [129]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [130]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 261 entries, 91 to 2559
Data columns (total 9 columns):
Lemma                    261 non-null object
Word_Type                261 non-null object
Category                 261 non-null object
Primary_subcategory      261 non-null object
Secondary_subcategory    51 non-null object
Sg_all_split             261 non-null object
Pl_all                   261 non-null object
Pl_all_split             261 non-null object
Alt_split                261 non-null object
dtypes: object(9)
memory usage: 20.4+ KB


### Check if subcategory name is missing and set the parent category name to subcategory if the subategory name is missing¶

In [131]:
#check missing values in Secondary subcategory
df_gendered[df_gendered['Secondary_subcategory'].isna()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
91,Congressman,s,gendered,titles,NaN,"[Member of the House, representative, legislat...",Members of the House | legislators | Senators ...,"[Members of the House, legislators, Senators, ...","[Member of the House, representative, legislat..."
92,Congresswoman,s,gendered,titles,NaN,"[Member of the House, representative, legislat...",Members of the House | legislators | Senators ...,"[Members of the House, legislators, Senators, ...","[Member of the House, representative, legislat..."
111,Dear Sir,i,gendered,function,NaN,"[Dear ((name or professional title)), Dear ((n...",,[],"[Dear ((name or professional title)), Dear ((n..."
197,Karen,s,gendered,female_stereotype,NaN,[someone who uses their privilege to get their...,people who use their privileges to demand thei...,[people who use their privileges to demand the...,[someone who uses their privilege to get their...
421,abuser,s,gendered,function,NaN,"[abusive person(...), someone who preys on oth...",,[],"[abusive person(...), someone who preys on oth..."
424,account manager,s,gendered,titles,NaN,"[person in account management, they, account m...",people in account management | account managem...,"[people in account management, account managem...","[account management, person in account managem..."
455,alpha,s,gendered,male_stereotype,NaN,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
456,alpha animal,s,gendered,male_stereotype,NaN,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
457,alpha dog,s,gendered,male_stereotype,NaN,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
458,alpha person,s,gendered,male_stereotype,NaN,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."


In [132]:
#fill the missing values in "Secondary subcategory" by values from "Primary_Category"
df_gendered.Secondary_subcategory.fillna(df_gendered.Primary_subcategory, inplace=True)

/Users/elenanazarenko/opt/anaconda3/envs/py38/lib/python3.8/site-packages/pandas/core/generic.py:6287: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._update_inplace(new_data)


In [133]:
#check missing values in Secondary subcategory
df_gendered[df_gendered['Secondary_subcategory'].isna()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [134]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_binary_pronouns(df_gendered)

In [135]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 169 entries, 0 to 168
Data columns (total 9 columns):
Lemma                    169 non-null object
Word_Type                169 non-null object
Category                 169 non-null object
Primary_subcategory      169 non-null object
Secondary_subcategory    169 non-null object
Sg_all_split             169 non-null object
Pl_all                   169 non-null object
Pl_all_split             169 non-null object
Alt_split                169 non-null object
dtypes: object(9)
memory usage: 12.0+ KB


In [136]:
df_gendered_word[df_gendered_word["Secondary_subcategory"].notnull()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Congressman,s,gendered,titles,titles,"[Member of the House, representative, legislat...",Members of the House | legislators | Senators ...,"[Members of the House, legislators, Senators, ...","[Member of the House, representative, legislat..."
1,Congresswoman,s,gendered,titles,titles,"[Member of the House, representative, legislat...",Members of the House | legislators | Senators ...,"[Members of the House, legislators, Senators, ...","[Member of the House, representative, legislat..."
2,Karen,s,gendered,female_stereotype,female_stereotype,[someone who uses their privilege to get their...,people who use their privileges to demand thei...,[people who use their privileges to demand the...,[someone who uses their privilege to get their...
3,abuser,s,gendered,function,function,"[abusive person(...), someone who preys on oth...",,[],"[abusive person(...), someone who preys on oth..."
4,actress,s,gendered,titles,generic_plural,"[actor, performer, they, actors, performers]",actors | performers,"[actors, performers]","[actor, performer, they, actors, performers]"
5,administrator,s,gendered,titles,generic_plural,"[they, administration, administrative team, ad...",administrators | administrative team | adminis...,"[administrators, administrative team, administ...","[administration, they, administrative team, ad..."
6,agent,s,gendered,titles,generic_plural,"[representative, they, agency, management, rep...",representatives | agents | agency | management...,"[representatives, agents, agency, management, ...","[agency, management, representation, represent..."
7,alpha,s,gendered,male_stereotype,male_stereotype,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
8,alumna,s,gendered,function,function,"[graduate, graduates]",graduates,[graduates],"[graduate, graduates]"
9,alumnu,s,gendered,function,function,"[graduate, graduates]",graduates,[graduates],"[graduate, graduates]"


In [137]:
df_gendered_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 9 columns):
Lemma                    92 non-null object
Word_Type                92 non-null object
Category                 92 non-null object
Primary_subcategory      92 non-null object
Secondary_subcategory    92 non-null object
Sg_all_split             92 non-null object
Pl_all                   92 non-null object
Pl_all_split             92 non-null object
Alt_split                92 non-null object
dtypes: object(9)
memory usage: 6.6+ KB


In [138]:
df_gendered_sentence[df_gendered_sentence["Secondary_subcategory"].notnull()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
0,Dear Sir,i,gendered,function,function,"[Dear ((name or professional title)), Dear ((n...",,[],"[Dear ((name or professional title)), Dear ((n..."
1,account manager,s,gendered,titles,titles,"[person in account management, they, account m...",people in account management | account managem...,"[people in account management, account managem...","[account management, person in account managem..."
2,alpha animal,s,gendered,male_stereotype,male_stereotype,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
3,alpha dog,s,gendered,male_stereotype,male_stereotype,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
4,alpha person,s,gendered,male_stereotype,male_stereotype,"[facilitator, mentor, supporter, facilitators,...",facilitators | mentors | supporters | people w...,"[facilitators, mentors, supporters, people who...","[facilitator, mentor, supporter, person who su..."
5,attention seeking,=a,gendered,female_stereotype,female_stereotype,"[outgoing, at ease in social situations, grega...",,[],"[outgoing, at ease in social situations, grega..."
6,back end developer,s,gendered,titles,generic_plural,"[they, back-end development, back-end developm...",back-end developers | back-end development tea...,"[back-end developers, back-end development tea...","[back-end development, they, colleague in back..."
7,back-end developer,s,gendered,titles,generic_plural,"[they, back-end development, back-end developm...",back-end developers | back-end development tea...,"[back-end developers, back-end development tea...","[back-end development, they, colleague in back..."
8,backend developer,s,gendered,titles,generic_plural,"[they, back-end development, back-end developm...",back-end developers | back-end development tea...,"[back-end developers, back-end development tea...","[back-end development, they, colleague in back..."
9,big boy,i,gendered,leadership,leadership,"[higher-up, supervisor, executive, management,...",management | administration,"[management, administration]","[management, administration, higher-up, superv..."


In [139]:
df_gendered["Secondary_subcategory"].unique()

array(['titles', 'function', 'female_stereotype', 'generic_plural',
       'male_stereotype', 'leadership', 'hidden_image', 'binary_pronouns'],
      dtype=object)

In [140]:
df_gendered["Primary_subcategory"].unique()

array(['titles', 'function', 'female_stereotype', 'male_stereotype',
       'leadership', 'hidden_image', 'binary_pronouns'], dtype=object)

In [141]:
df_gendered_word.query("Word_Type == '=s'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
71,histrionics,=s,gendered,female_stereotype,female_stereotype,"[histrionic personality disorder, person with ...",histrionic personality disorder,[histrionic personality disorder],"[histrionic personality disorder, person with ..."
72,housekeeping,=s,gendered,hidden_image,hidden_image,"[maintenance, cleanup]",,[],"[maintenance, cleanup]"


In [142]:
df_gendered_word.query("Word_Type == '-s'")

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [143]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's' or Word_Type == '=s'")

In [144]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 149 entries, 0 to 167
Data columns (total 9 columns):
Lemma                    149 non-null object
Word_Type                149 non-null object
Category                 149 non-null object
Primary_subcategory      149 non-null object
Secondary_subcategory    149 non-null object
Sg_all_split             149 non-null object
Pl_all                   149 non-null object
Pl_all_split             149 non-null object
Alt_split                149 non-null object
dtypes: object(9)
memory usage: 11.6+ KB


In [145]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [146]:
df_gendered_word_no_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 22 entries, 14 to 168
Data columns (total 9 columns):
Lemma                    22 non-null object
Word_Type                22 non-null object
Category                 22 non-null object
Primary_subcategory      22 non-null object
Secondary_subcategory    22 non-null object
Sg_all_split             22 non-null object
Pl_all                   22 non-null object
Pl_all_split             22 non-null object
Alt_split                22 non-null object
dtypes: object(9)
memory usage: 1.7+ KB


In [147]:
df_gendered_word_no_noun = df_gendered_word_no_noun.query("Word_Type != '=s'")

In [148]:
df_gendered_word_no_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 20 entries, 14 to 168
Data columns (total 9 columns):
Lemma                    20 non-null object
Word_Type                20 non-null object
Category                 20 non-null object
Primary_subcategory      20 non-null object
Secondary_subcategory    20 non-null object
Sg_all_split             20 non-null object
Pl_all                   20 non-null object
Pl_all_split             20 non-null object
Alt_split                20 non-null object
dtypes: object(9)
memory usage: 1.6+ KB


In [149]:
#drop unnecessary column "Pl_all"
df_gendered_word_noun = df_gendered_word_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all"], axis=1)

In [150]:
#save to csv file after preprocessing
df_gendered_word_noun.to_csv("gendered_noun_words.csv", index=False)

In [151]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words.csv", index=False)

In [152]:
df_gendered_sentence.to_csv("gendered_sentences.csv", index=False)

In [153]:
replace_tolist(r"gendered_noun_words.csv")

'gendered_noun_words.csv'

In [154]:
replace_tolist(r"gendered_no_noun_words.csv")

'gendered_no_noun_words.csv'

In [155]:
replace_tolist(r"gendered_sentences.csv")

'gendered_sentences.csv'

## Unconscious Bias

In [156]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [157]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 898 entries, 6 to 2613
Data columns (total 9 columns):
Lemma                    898 non-null object
Word_Type                898 non-null object
Category                 898 non-null object
Primary_subcategory      898 non-null object
Secondary_subcategory    219 non-null object
Sg_all_split             898 non-null object
Pl_all                   898 non-null object
Pl_all_split             898 non-null object
Alt_split                898 non-null object
dtypes: object(9)
memory usage: 70.2+ KB


In [158]:
#check missing values in Secondary subcategory
df_ub[df_ub['Secondary_subcategory'].notnull()]

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
22,Are you blind?,i,unconscious_bias,vision,cultural,"[Don't you see?, Don't you realize?, Didn't yo...",,[],"[Don't you see?, Don't you realize?, Didn't yo..."
353,See you soon,i,unconscious_bias,vision,cultural,"[Be in touch soon, Hope to connect soon, Hope ...",,[],"[Be in touch soon, Hope to connect soon, Hope ..."
415,abort,v,unconscious_bias,agentic,tech,"[terminate, stop, cancel, pause, suspend]",,[],"[terminate, stop, cancel, pause, suspend]"
425,achiever,s,unconscious_bias,agentic,competitive,"[person who sees things through(...), person w...",people who see things through(...) | those wit...,"[people who see things through(...), those wit...","[person who sees things through(...), person w..."
432,adventurous,a,unconscious_bias,agentic,competitive,"[enterprising, resourceful, happy to explore n...",,[],"[enterprising, resourceful, happy to explore n..."
441,aggression,s,unconscious_bias,agentic,hard_hearted,"[commitment to achieving a goal, focus on purp...",positive attitudes toward using one's competen...,[positive attitudes toward using one's compete...,"[commitment to achieving a goal, focus on purp..."
442,aggressive,a,unconscious_bias,agentic,hard_hearted,"[energetic, emphatic, invested, spirited, open...",,[],"[energetic, emphatic, invested, spirited, open..."
454,alone,a,unconscious_bias,agentic,competitive,"[uninterrupted, without distractions, --- don'...",--- don't use for people,[--- don't use for people],"[--- don't use for people, uninterrupted, with..."
462,ambition,s,unconscious_bias,agentic,competitive,"[growth mindset, dedication, willingness to go...",growth mindset | dedication | willingness to g...,"[growth mindset, dedication, willingness to go...","[growth mindset, dedication, willingness to go..."
463,ambitious,a,unconscious_bias,agentic,competitive,"[growth-minded, keen and enthusiastic, --- don...",--- don't use for people,[--- don't use for people],"[--- don't use for people, growth-minded, keen..."


In [159]:
#fill the missing values in "Secondary subcategory" by values from "Primary_Category"
df_ub.Secondary_subcategory.fillna(df_ub.Primary_subcategory, inplace=True)

In [160]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences(df_ub)

In [161]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 431 entries, 0 to 430
Data columns (total 9 columns):
Lemma                    431 non-null object
Word_Type                431 non-null object
Category                 431 non-null object
Primary_subcategory      431 non-null object
Secondary_subcategory    431 non-null object
Sg_all_split             431 non-null object
Pl_all                   431 non-null object
Pl_all_split             431 non-null object
Alt_split                431 non-null object
dtypes: object(9)
memory usage: 30.4+ KB


In [162]:
df_ub_word[df_ub_word["Word_Type"]=='-s']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split


In [163]:
df_ub_word[df_ub_word["Word_Type"]=='=s']

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split
4,IQ-test,=s,unconscious_bias,cognitive_perception,cognitive_perception,"[aptitude test, deliberative test, learning sk...",aptitude tests | deliberative tests | learning...,"[aptitude tests, deliberative tests, learning ...","[aptitude test, deliberative test, learning sk..."
5,IQtest,=s,unconscious_bias,cognitive_perception,cognitive_perception,"[aptitude test, deliberative test, learning sk...",aptitude tests | deliberative tests | learning...,"[aptitude tests, deliberative tests, learning ...","[aptitude test, deliberative test, learning sk..."
245,kinsmen,=s,unconscious_bias,gender_identity,gender_identity,[relatives],,[],[relatives]


In [164]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [165]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 148 entries, 0 to 429
Data columns (total 9 columns):
Lemma                    148 non-null object
Word_Type                148 non-null object
Category                 148 non-null object
Primary_subcategory      148 non-null object
Secondary_subcategory    148 non-null object
Sg_all_split             148 non-null object
Pl_all                   148 non-null object
Pl_all_split             148 non-null object
Alt_split                148 non-null object
dtypes: object(9)
memory usage: 11.6+ KB


#### save the left data into separate dataframe

In [166]:
plural_list = list(df_ub_word_pl["Lemma"])

In [167]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all" 

In [168]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all"], axis=1)

In [169]:
# replace secondary_subcategory by primary_subcategory for ub_noun because it not used by the moment
df_ub_word_pl.Secondary_subcategory=df_ub_word_pl.Primary_subcategory

In [170]:
df_ub_word_pl.head()

,Lemma,Word_Type,Category,Primary_subcategory,Secondary_subcategory,Sg_all_split,Pl_all_split,Alt_split
0,Asperger,s,unconscious_bias,ability,ability,"[Autistic, neurodivergent, on the autism spect...","[--- only if the person prefers the term, or w...","[--- only if the person prefers the term, or w..."
1,Cervix-haver,s,unconscious_bias,gender_identity,gender_identity,[person with a cervix(...)],[people with a cervix(...)],[person with a cervix(...)]
2,Cervixhaver,s,unconscious_bias,gender_identity,gender_identity,[person with a cervix(...)],[people with a cervix(...)],[person with a cervix(...)]
3,Eskimo,s,unconscious_bias,racist_source,racist_source,"[Inuk, Aleut, Athabascan, Cup’ik, Haida, Iñupi...","[Inuit, Alutiit, Athabascans, Cup’ik, Haida, I...","[Inuk, Aleut, Athabascan, Cup’ik, Haida, Iñupi..."
6,Indian,s,unconscious_bias,racist_source,racist_source,"[First Nation, Native American, BIPoC, Native ...","[First Nations, Native Americans, BIPoC, Indig...","[--- name the First Nation (Americas), First N..."


#### save to csv files after preprocessing

In [171]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words.csv", index=False)
df_ub_sentence.to_csv("ub_sentences.csv", index=False)

In [172]:
replace_tolist(r"ub_no_plur_words.csv")

'ub_no_plur_words.csv'

In [173]:
replace_tolist(r"ub_plur_words.csv")

'ub_plur_words.csv'

In [174]:
replace_tolist(r"ub_sentences.csv")

'ub_sentences.csv'